# Simple functions for ground and excited states

## Example: the electronic states of molecular oxygen

This example notebook shows how to determine active spaces for the triplet ground state of oxygen, and its lowest excited singlet states. An emphasis is placed on simple wrapper functions to determine active spaces directly from a molecule definition.

The following learning points are covered:

- Selecting an active space using a simple wrapper function.
- Selecting an active space using a wrapper function for multiple electronic states.
- Using the selected active space in CASSCF calculations, including an example with state averaging.
- Using suitable wrapper functions to choose active spaces of a specific size defined by the user.

In [ ]:
# Suppress PySCF warning...
import pyscf
pyscf.__config__.B3LYP_WITH_VWN5 = False

In [ ]:
from pathlib import Path

# The Mole class is used to define molecular information in PySCF.
from pyscf.gto import Mole

# logger contains definitions of verbosity levels for PySCF.
from pyscf.lib import logger

# Functionality for (state-averaged) CASSCF.
from pyscf.mcscf import CASSCF, state_average_mix
from pyscf.fci.direct_spin1 import FCISolver
from pyscf.fci.addons import fix_spin

# Wrapper functions to perform selection for variable and fixed active space sizes
from asf.wrapper import find_from_mol, find_from_scf, sized_space_from_mol, sized_space_from_scf

# Various utility functions...
from asf.utility import compare_active_spaces, show_mos_grid, pictures_Jmol

There are two variants of the wrapper function to find general active spaces: `find_from_mol` and `find_from_scf`. The former accepts a `Mole` objects containing a molecular definition and performs the entire active space selection, including an SCF calculation. It is usually performed as a UHF calculation even if the molecule does not have any unpaired electrons. Stability analysis is used to help the UHF calculation converge to a broken-symmetry solution, which helps with the initial active space selection. It is also possible to start from a converged RHF or UHF solution directly using `find_from_scf`, which carries out the same steps as `find_from_mol` with the exception of the SCF calculation itself.

In [ ]:
help(find_from_mol)

In [ ]:
help(find_from_scf)

We start with a molecule definition of oxygen in its (triplet) ground state.

In [ ]:
mol = Mole()
mol.atom = "O2.xyz"
mol.basis = "def2-SVP"
mol.charge = 0
mol.spin = 2
# Set mol.verbose = logger.INFO to enable printing of SCF iterations and further output.
mol.verbose = logger.NOTE
mol.build()

All that is required now for an active space selection is one function call:

In [ ]:
active_space = find_from_mol(mol)

The returned object contains the number of active MOs and electrons, the matrix of MO coeffients and a list of active orbital indices referring to the aforementioned matrix.

In [ ]:
print(active_space)

Below, we plot the active orbitals that were found.

In [ ]:
pictures_Jmol(
    mol=mol, mo_coeff=active_space.mo_coeff, mo_list=active_space.mo_list, mo_index_title=" "
)
mo_images = sorted(dd for dd in Path.cwd().glob("mo_*.png"))
show_mos_grid(mo_images, mo_list=active_space.mo_list, columns=3, figsize=(10, 10))
# clean up files
for filename in mo_images:
    Path.unlink(filename)

The selected active space can be used to set up a CASSCF calculation. Note that it is important to use the orbitals returned by the ASF in the object `active_space` - not the orbitals obtained from SCF or elsewhere!

In [ ]:
mc = CASSCF(mol, ncas=active_space.norb, nelecas=active_space.nel)
# Before carrying out the CASSCF calculation, the orbital coefficient matrix columns need to be
# ordered as core - active - virtual from left to right.
# Take care - by default, sort_mo assumes that the MO list starts indexing from one, not from zero.
mo_guess = mc.sort_mo(active_space.mo_list, active_space.mo_coeff, base=0)
_ = mc.kernel(mo_coeff=mo_guess)

A CASSCF calculation converging in a small number of iterations is often a good sign. In general, it is advisable to inspect converged CASSCF orbitals visually in order to verify that they indeed form the expected active space. In our case, that would mean comparing them to the guess orbitals determined by the active space finder.

Here, however, we will employ a simple numerical criterion by performing a singular value decomposition of the overlap matrix between the guess and the converged active orbitals. The following function reports the smallest singular value - if it is close to one, that is a good sign. A value close to zero indicates that an orbital may have rotated out, while intermediate values indicate some degree of mixing.

In [ ]:
compare_active_spaces(mc, active_space.mo_coeff, active_space.mo_list)

The functions `find_from_mol` / `find_from_scf` can also be employed to find active spaces for state-averaged CASSCF calculations of multiple electronic states. This functionality is relatively simple, determining active spaces for each electronic state in isolation, and then forming the union of the MO sets. States can be specified in one of two ways:

1. Using `states: int`. in that case, the active space will be determined for a number `states` of states with the multiplicity inferred from the Mole or SCF object.
2. Using `states: list[tuple[int, int]]`. the first integer in each tuple specifies the number of electrons (`spin` in PySCF conventions), the second integer the number of electronic states of that spin. In this case, the union is formed over the active spaces for all states of all spin multiplicities requested.
Below this will be demonstrated for one triplet and three singlet states of oxygen:

In [ ]:
active_space = find_from_mol(mol, states=[(0, 3), (2, 1)])

The recommended active space is the same as for triplet oxygen; note that the table printed by the wrapper can be inspected for the orbital occupations in each state.

Below, a state-averaged CASSCF calculation is performed with four states: the triplet, the two doubly degenerate lowest singlet states, and the third singlet state.

In [ ]:
mc = CASSCF(mol, ncas=active_space.norb, nelecas=active_space.nel)
# The solvers for each spin need to be set up individually.
# See PySCF examples for state averages and state average mixes.
solver_t = FCISolver(mol)
solver_t.spin = 2
solver_s = FCISolver(mol)
solver_s.spin = 0
solver_s.nroots = 3
solver_s = fix_spin(solver_s, ss=0)
# All four states are weighted equally.
mc_mix = state_average_mix(mc, [solver_t, solver_s], weights=[0.25] * 4)
mo_guess = mc.sort_mo(active_space.mo_list, active_space.mo_coeff, base=0)
_ = mc_mix.kernel(mo_coeff=mo_guess)

The main parameter controlling the size of the selected space is the entropy threshold, controlled via the parameter `entropy_threshold` of the `find_from_mol` and `find_from_scf` functions.

If you know the size of the desired active space and only need to select the correct orbitals, it is possible to do so via the functions `sized_space_from_mol` and `sized_space_from_scf`. The only difference between the two functions is that the former performs an SCF calculation, while the latter expects an object with an SCF solution as its input. These wrapper functions currently support only one electronic state at a time.

In [ ]:
help(sized_space_from_mol)

In [ ]:
help(sized_space_from_scf)

It is possible to specify the size of an active space either via the number of orbitals by providing an integer, or by providing both the number of active electrons and orbitals as a tuple `(electrons, orbitals)`. If only the number of orbitals is provided, the number of electrons is determined by the active space finder.

For example, the following function call constrains the number of active orbitals to four and chooses an active space with a suitable number of electrons, resulting in a (6, 4) space:

In [ ]:
sized_space_from_mol(mol, size=4)

Likewise, it is possible to specify six electrons and four orbitals explicitly:

In [ ]:
sized_space_from_mol(mol, size=(6, 4))

Note that the `sized_space` functions attempt to choose active spaces from a set of sensible suggestions determined internally. In consequence, if the user requests an active space size that is not considered sensible, the function will raise an `ActiveSpaceSelectionError` exception.